- Statement of Academic Honesty: <br>
  The following answers represents my own work. I have neither received nor given inappropriate assistance. I have not copied or modified solutions from any source other than the course platform or the course textbook. Each section of the code is clearly marked to indicate whether it is human-generated (created by me) or machine-generated (created with the assistance of an AI tool). I recognize that any unauthorized assistance or plagiarism will be handled in accordance with the Northeastern University's Academic Honesty Policy and the policies of this course. Any publishing or posting of this exam is strictly prohibited.

- Initials: **XX**

- Submission Date: **11/19/2024**

In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime
from collections import Counter, defaultdict

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load the JSON dataset
def load_data(file_path): 
    with open(file_path, 'r') as f:
        return [json.loads(line) for line in f]

data = load_data('conti_conversations_embeddings_summaries.jsonl')

In [ ]:
# Extract summaries/embeddings for analysis
summaries = [record['summary'] for record in data]
embeddings = np.array([record['embeddings'] for record in data])
timestamps = [(record['start_time'], record['end_time']) for record in data]

## Question 1: Targeted Organizations 
Solution 1: Using summaries as they are likely to contain explicit textual mentions of organizations.

In [ ]:
def find_targeted_organizations(summaries):

    # Define patterns that might indicate organizations
    org_keywords = ["company", "organization", "firm"]
    targeted_orgs = []

    # Search for potential organization mentions
    for summary in summaries:
        for keyword in org_keywords:
            if keyword in summary.lower():
                # Extract possible organization names with simple regex
                org_matches = re.findall(r"[A-Z][a-zA-Z]+(?: [A-Z][a-zA-Z]+)?", summary)
                targeted_orgs.extend(org_matches)

    return Counter(targeted_orgs).most_common()

# Run the function on summaries
targeted_organizations = find_targeted_organizations(summaries)
print("Potentially Targeted Organizations:")
for org, count in targeted_organizations:
    print(f"{org}: {count}")

## Question 2: New Attack Planning 
Solution 2: Using summaries as they are more suitable for extracting explicit future planning details.

In [ ]:
# Function to identify new attack planning discussions
def detect_new_attack_plans(summaries):
    # Keywords/phrases related to planning new attacks
    attack_related_phrases = [
        "attack plan", 
        "new attack",
        "plan to attack", 
        "plans to attack", 
        "upcoming attack"
    ]
    
    # Filter summaries containing relevant phrases
    plans = [
        summary for summary in summaries 
        if any(phrase in summary.lower() for phrase in attack_related_phrases)
    ]
    
    return plans

# Identify discussions related to new attack plans
new_attack_plans = detect_new_attack_plans(summaries)

# Display the results
print("New Attack Planning Discussions:")
for i, plan in enumerate(new_attack_plans, start=1):
    print(f"{i}. {plan}")


## Question 3: Botnet Discussions
Solution 3: Using summaries to identify explicit mentions of botnets.

In [ ]:
# Function to identify botnet discussions
def find_botnet_mentions(summaries):
    # Keywords related to botnet discussions
    botnet_keywords = [
        "botnet", 
        "zombie"
    ]
    
    # Filter summaries containing relevant phrases
    botnet_discussions = [
        summary for summary in summaries 
        if any(keyword in summary.lower() for keyword in botnet_keywords)
    ]
    
    return botnet_discussions

# Identify discussions related to botnets
botnet_mentions = find_botnet_mentions(summaries)

# Display the results
print("Botnet Mentions:")
for i, mention in enumerate(botnet_mentions, start=1):
    print(f"{i}. {mention}")


## Question 4: Bitcoin Usage
Solution 4: Using summaries to search for mentions of bitcoin or cryptocurrency usage.

In [ ]:
# Function to identify bitcoin mentions
def find_bitcoin_mentions(summaries):
    # Keywords related to bitcoin or cryptocurrency
    bitcoin_keywords = [
        "bitcoin", 
        "btc "
    ]
    
    # Filter summaries containing relevant phrases
    bitcoin_discussions = [
        summary for summary in summaries 
        if any(keyword in summary.lower() for keyword in bitcoin_keywords)
    ]
    
    return bitcoin_discussions

# Identify discussions related to bitcoin
bitcoin_mentions = find_bitcoin_mentions(summaries)

# Display the results
print("Bitcoin Discussions:")
for i, mention in enumerate(bitcoin_mentions, start=1):
    print(f"{i}. {mention}")


## Question 5: Dynamic Topic Shift Detection
Solution 5: Using embeddings and timestamps for topic shift analysis.

In [ ]:
timestamps_datetime = [
    (datetime.fromtimestamp(start / 1000), datetime.fromtimestamp(end / 1000))
    for start, end in timestamps
]

def aggregate_daily_embeddings(timestamps_datetime, embeddings):
    daily_embeddings = defaultdict(list)
    for ts, emb in zip(timestamps_datetime, embeddings):
        date = ts[0].date()  # Use the start time's date
        daily_embeddings[date].append(emb)
    
    # Compute daily mean embeddings
    aggregated_embeddings = {date: np.mean(np.array(emb_list), axis=0) 
                              for date, emb_list in daily_embeddings.items()}
    return aggregated_embeddings

# Aggregate daily embeddings
daily_embeddings = aggregate_daily_embeddings(timestamps_datetime, embeddings)
dates = sorted(daily_embeddings.keys())
aggregated_embeddings = np.array([daily_embeddings[date] for date in dates])

# Detect daily topic shifts
def detect_daily_topic_shifts(aggregated_embeddings, threshold=0.9):
    similarities = []
    shifts = []
    for i in range(1, len(aggregated_embeddings)):
        sim = cosine_similarity(aggregated_embeddings[i - 1].reshape(1, -1),
                                 aggregated_embeddings[i].reshape(1, -1))[0, 0]
        similarities.append(sim)
        # Detect shift if similarity is below the threshold
        if sim < threshold:
            shifts.append((i, dates[i]))  # Store index and date of the shift
    return similarities, shifts

daily_similarities, topic_shifts = detect_daily_topic_shifts(aggregated_embeddings)

# Visualize daily topic shifts
def plot_daily_topic_shifts(dates, daily_similarities, topic_shifts, threshold=0.9):
    plt.figure(figsize=(12, 6))
    plt.plot(dates[1:], daily_similarities, marker='o', label="Cosine Similarity", markersize=5)
    plt.axhline(threshold, color="gray", linestyle="--", label="Threshold")
    for shift in topic_shifts:
        plt.axvline(shift[1], color="red", linestyle="--", alpha=0.6, label="Topic Shift" if shift == topic_shifts[0] else "")
    plt.title("Daily Topic Shifts")
    plt.xlabel("Date")
    plt.ylabel("Cosine Similarity")
    plt.xticks(rotation=45)
    plt.legend()
    plt.show()

plot_daily_topic_shifts(dates, daily_similarities, topic_shifts)


By aggregating embeddings for each day and calculating cosine similarities between consecutive days, we focus on detect moments where the main topic changed substantially and marked it with a red line in the above garph.